# 02 Database Connection

## Goal

The goal of this notebook is to connect to the PostgreSQL database and validate that the Steam dataset tables are available.

At this stage, we only check:
- database connection,
- available tables,
- row counts,
- consistency with previous CSV overview.

No cleaning, feature engineering or modeling is performed yet.

## Load Libraries

In [2]:
import os
from idlelib import query
from unittest import result

import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect, text

## Load Environment Variables

In [3]:
PROJECT_ROOT = Path("..")
load_dotenv(PROJECT_ROOT / ".env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_SCHEMA = os.getenv("DB_SCHEMA", "steam")

## Create Database Connection

In [5]:
connection_string =(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
engine = create_engine(connection_string)

In [6]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT version();"))
    print(result.fetchone()[0])

PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit


## List Available Tables

In [8]:
inspector = inspect(engine)
tables = inspector.get_table_names(schema=DB_SCHEMA)

tables

['games',
 'achievements',
 'history',
 'prices',
 'players',
 'friends',
 'private_steamids',
 'purchased_games',
 'reviews']

## Tables Overview

In [10]:
db_overview = []

with engine.connect() as connection:
    for table_name in tables:
        query = text(f'SELECT COUNT(*) FROM "{DB_SCHEMA}"."{table_name}"')
        row_count = connection.execute(query).scalar()

        db_overview.append({
            "table_name": table_name,
            "row_count": row_count
        })

db_overview_df = pd.DataFrame(db_overview).sort_values(
    by="row_count",
    ascending=False
)

db_overview_df

,table_name,row_count
2,history,125250422
3,prices,4414273
1,achievements,1939027
8,reviews,1204534
4,players,424683
5,friends,424683
6,private_steamids,227963
7,purchased_games,102548
0,games,98465


In [13]:
PROCESSED_DATA_PATH = PROJECT_ROOT / "Data" / "processed"

In [14]:
db_overview_df.to_csv(
    PROCESSED_DATA_PATH / "database_table_overview.csv",
    index=False
)

In [15]:
column_overview = []

for table_name in tables:
    columns = inspector.get_columns(table_name, schema=DB_SCHEMA)

    for column in columns:
        column_overview.append({
            "table_name": table_name,
            "column_name": column["name"],
            "data_type": str(column["type"]),
            "nullable": column["nullable"]
        })

column_overview_df = pd.DataFrame(column_overview)

column_overview_df

,table_name,column_name,data_type,nullable
0,games,game_id,INTEGER,False
1,games,title,TEXT,False
2,games,developers,ARRAY,True
3,games,publishers,ARRAY,True
4,games,genres,ARRAY,True
5,games,supported_languages,ARRAY,True
6,games,release_date,DATE,True
7,achievements,achievement_id,TEXT,False
8,achievements,game_id,INTEGER,False
9,achievements,title,TEXT,False


In [16]:
column_overview_df.to_csv(
    PROCESSED_DATA_PATH / "database_column_overview.csv",
    index=False
)